# Phase 36 — Robustness under image perturbation

Reconstructed notebook for `phase_36_robustness_perturbation`. The scientific execution (v1.23) produced 12,928 perturbation instances over 808 clean images.

> **Status:** reconstructed analysis notebook. It reads persisted artifacts only. No training, no new inference, no protected TEST access.

In [1]:
import sys
from pathlib import Path
sys.path.insert(0, r"D:\Softfors\jup\full_doc_report\Scripts")
import remedfilm_guard as GUARD
GUARD.install()

REPO = Path(r"D:\Softfors\jup")
REPORT = REPO / "full_doc_report"
TABLES = REPORT / "tables"
FIG = REPORT / "figures"

import json
import numpy as np
import pandas as pd
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
from IPython.display import Image, display, Markdown

def fig(subdir, name, width=760):
    """Display a generated figure, failing loudly if it is missing."""
    p = FIG / subdir / f"{name}.png"
    if not p.exists():
        raise FileNotFoundError(f"missing figure: {p}")
    display(Image(filename=str(p), width=width))

def table(path, **kw):
    return pd.read_csv(path, **kw)

GUARD.require()

True

In [2]:
# Negative control — enforcement must be real, not a flag.
for name, verdict in GUARD.self_test():
    print(f"  {name:28s}: {verdict}")
assert all(v.startswith("blocked") for _, v in GUARD.self_test()), "GUARD FAILED"
print("analysis-only guard verified")

  optimiser construction      : blocked
  AdamW construction          : blocked
  TEST manifest read          : blocked
  write outside report dir    : blocked
analysis-only guard verified


In [3]:
r = json.loads((REPO / "reports/audits/phase36_robustness_scientific_result_v1_23.json").read_text(encoding="utf-8"))
print("counters")
for k in ("clean_passes", "perturbed_g_forwards", "test_reads", "training",
          "optimizer_steps", "parameter_mutations"):
    print(f"  {k:22s}: {r['counters'][k]}")
print()
print("clean equivalence all_pass:", r["clean_equivalence"]["all_pass"])

counters
  clean_passes          : 808
  perturbed_g_forwards  : 12928
  test_reads            : 0
  training              : 0
  optimizer_steps       : 0
  parameter_mutations   : 0

clean equivalence all_pass: True


In [4]:
rows = pd.DataFrame(r["image_rows"])
print("perturbation rows:", len(rows), "| conditions:", rows.condition.nunique())
agg = (rows.groupby("condition")
           .agg(n=("paired_delta_dice", "size"),
                mean_delta=("paired_delta_dice", "mean"),
                min_delta=("paired_delta_dice", "min"),
                consistency=("prediction_consistency_dice", "mean"))
           .sort_values("mean_delta").reset_index())
agg.round(5)

perturbation rows:

 12928 | conditions: 12


,condition,n,mean_delta,min_delta,consistency
0,GAUSSIAN_NOISE_HIGH,2424,-0.15579,-0.92113,0.72882
1,GAUSSIAN_NOISE_LOW,2424,-0.04286,-0.87407,0.88730
2,GAUSSIAN_BLUR_HIGH,808,-0.02357,-0.77778,0.92203
3,RESOLUTION_050,808,-0.02307,-0.77778,0.92338
4,RESOLUTION_075,808,-0.01361,-0.77778,0.95163
5,BRIGHTNESS_NEG,808,-0.00765,-0.77698,0.95705
6,BRIGHTNESS_POS,808,-0.00627,-0.65010,0.95997
7,GAMMA_LOW,808,-0.00494,-0.50907,0.96618
8,CONTRAST_LOW,808,-0.00429,-0.36443,0.96753
9,GAUSSIAN_BLUR_LOW,808,-0.00244,-0.23694,0.98017


In [5]:
fig_, ax = plt.subplots(figsize=(7.0, 2.8))
colors = ["#D55E00" if v < -0.05 else ("#E69F00" if v < 0 else "#009E73") for v in agg.mean_delta]
ax.bar(range(len(agg)), agg.mean_delta, color=colors, width=0.68)
ax.axhline(0, color="#333333", linewidth=0.8)
ax.set_xticks(range(len(agg)))
ax.set_xticklabels([c.replace("_", " ").lower() for c in agg.condition], rotation=40,
                   ha="right", fontsize=6)
ax.set_ylabel("Mean Δ Dice vs clean")
ax.set_title("Phase 36 — 12,928 perturbation instances, 12 conditions")
ax.grid(axis="x", visible=False); plt.tight_layout(); plt.show()

C:\Users\user\AppData\Local\Temp\ipykernel_36884\2872117821.py:10: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  ax.grid(axis="x", visible=False); plt.tight_layout(); plt.show()


In [6]:
# Domain-specific degradation — the KUB finding.
by_dom = (rows.groupby(["domain", "condition"]).paired_delta_dice.mean()
              .unstack("condition"))
kub = by_dom.loc["KUB"].sort_values()
print("KUB worst conditions:")
print(kub.head(5).round(4).to_string())

KUB worst conditions:
condition
GAUSSIAN_NOISE_HIGH   -0.3636
GAUSSIAN_NOISE_LOW    -0.1153
GAUSSIAN_BLUR_HIGH    -0.0563
RESOLUTION_050        -0.0544
RESOLUTION_075        -0.0342


In [7]:
print("NEGATIVE FINDING PRESERVED")
print("--------------------------")
print("KUB degrades far more than the other domains under noise and resolution")
print("loss. This is a real weakness of the FULL system, not an artefact.")

NEGATIVE FINDING PRESERVED
--------------------------
KUB degrades far more than the other domains under noise and resolution
loss. This is a real weakness of the FULL system, not an artefact.


## Provenance

- `reports/audits/phase36_robustness_scientific_result_v1_23.json`

In [8]:
# Negative control — enforcement must be real, not a flag.
for name, verdict in GUARD.self_test():
    print(f"  {name:28s}: {verdict}")
assert all(v.startswith("blocked") for _, v in GUARD.self_test()), "GUARD FAILED"
print("analysis-only guard verified")

  optimiser construction      : blocked
  AdamW construction          : blocked
  TEST manifest read          : blocked
  write outside report dir    : blocked
analysis-only guard verified
